In [33]:
import pandas as pd
import numpy as np
from scipy.spatial import distance

In [34]:
# Load detections from CSV
df = pd.read_csv("data/rugby_detection_raw.csv")

In [35]:
# Extract numeric frame number
df["frame_num"] = df["frame"].str.extract(r"(\d+)").astype(int)

# Sort data by frame number
df = df.sort_values(by=["frame_num"])

# Dictionary to track player positions across frames
player_tracker = {}
player_id = 1
thresh = 10

# Assign unique player IDs ensuring no duplicate IDs per frame
df["player_id"] = np.nan

for frame in df["frame_num"].unique():
    frame_data = df[df["frame_num"] == frame].copy()
    used_ids = set()  # Track assigned IDs for this frame

    for index, row in frame_data.iterrows():
        if row["object"] != "person":
            continue

        team = row["team"]
        x, y = row["x_field"], row["y_field"]

        # Find closest match in the previous frame
        min_dist = float("inf")
        assigned_id = None

        for pid, (px, py, pteam) in player_tracker.items():
            if pteam == team and pid not in used_ids:  # Ensure uniqueness in the frame
                dist = distance.euclidean((x, y), (px, py))
                if dist < min_dist and dist < thresh:  # Distance threshold for tracking
                    min_dist = dist
                    assigned_id = pid

        # If no match found, assign a new ID
        if assigned_id is None:
            assigned_id = player_id
            player_id += 1

        df.at[index, "player_id"] = assigned_id
        player_tracker[assigned_id] = (x, y, team)
        used_ids.add(assigned_id)  # Ensure unique assignment in the frame

# Ensure player_id is properly set as integer
df["player_id"] = df["player_id"].astype("Int64")


In [36]:
df[df["player_id"] == 1]

,x_field,y_field,team,object,color,frame,frame_num,player_id
0,22.053741,3.827706,A,person,"[123, 58, 43]",frames_field/field_0000,0,1
17,19.248942,3.039323,A,person,"[144, 64, 53]",frames_field/field_0001,1,1


In [ ]:
# Update the original processing code to use player_id tracking

# Compute previous positions per player
df["prev_x"] = df.groupby("player_id")["x_field"].shift(1)
df["prev_y"] = df.groupby("player_id")["y_field"].shift(1)

# Compute velocities
df["velocity_x"] = df["x_field"] - df["prev_x"]
df["velocity_y"] = df["y_field"] - df["prev_y"]

# Define field dimensions
FIELD_WIDTH = 120  # X-axis
FIELD_HEIGHT = 70  # Y-axis

# Identify the attacking player closest to the ball in each frame

# Initialize the column
df["attacker_with_ball"] = np.nan

# Iterate over each frame
for frame in df["frame_num"].unique():
    frame_data = df[df["frame_num"] == frame]
    ball_row = frame_data[frame_data["object"] == "ball"]

    if not ball_row.empty:
        ball_x, ball_y = ball_row.iloc[0][["x_field", "y_field"]]

        # Filter for only attacking team players
        attacking_players = frame_data[(frame_data["object"] == "person") & (frame_data["team"].notna())]

        if not attacking_players.empty:
            # Compute distances to the ball
            attacking_players["ball_distance"] = np.sqrt(
                (attacking_players["x_field"] - ball_x) ** 2 + (attacking_players["y_field"] - ball_y) ** 2
            )

            # Find the player closest to the ball
            closest_player_id = attacking_players.loc[attacking_players["ball_distance"].idxmin(), "player_id"]

            # Mark the player as having the ball
            df.loc[(df["frame_num"] == frame) & (df["player_id"] == closest_player_id), "attacker_with_ball"] = 1

# Fill NaN with 0 for easier interpretation (1 = has ball, 0 = does not have ball)
df["attacker_with_ball"] = df["attacker_with_ball"].fillna(0).astype(int)

# Save structured dataset
df.to_csv("data/rugby_detection.csv", index=False)

print("✅ Rugby feature dataset saved as rugby_detection.csv!")

✅ Rugby feature dataset saved as rugby_detection.csv!


C:\Users\mthui\AppData\Local\Temp\ipykernel_3292\133336070.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attacking_players["ball_distance"] = np.sqrt(
C:\Users\mthui\AppData\Local\Temp\ipykernel_3292\133336070.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attacking_players["ball_distance"] = np.sqrt(


In [38]:
df[df["player_id"] == 1]

,x_field,y_field,team,object,color,frame,frame_num,player_id,prev_x,prev_y,velocity_x,velocity_y,ball_x,ball_y,ball_distance,goal_distance,angle_ball_goal,cos_angle,sin_angle,attacker_with_ball
0,22.053741,3.827706,A,person,"[123, 58, 43]",frames_field/field_0000,0,1,NaN,NaN,NaN,NaN,27.461341,10.460092,8.557492,102.787069,0.259219,0.966590,0.256326,0
24,19.248942,3.039323,A,person,"[144, 64, 53]",frames_field/field_0001,1,1,22.053741,3.827706,-2.804799,-0.788383,25.325881,25.879240,23.634529,105.698914,0.096042,0.995392,0.095895,0
